# KYC agent with Unity Catalog tools

BankBricks onboarding agent: collects name and account choice, verifies identity (passport) and proof of address, runs PEP/sanctions/sanctioned-country/fraud/SBI checks in the background, and opens requested accounts.

**Prerequisites:** Set **CATALOG** (and **SCHEMA**) in **Configuration**, then run the optional **Run setup** cell—it calls **`01_kyc_data_setup`** with those values. You can also run the setup notebook separately with the same catalog/schema.

---

## Overview: tools

| Tool | Type | Purpose |
|------|------|---------|
| `extract_kyc(first_name, last_name)` | Python | Look up and parse passport by person name -> first_name, last_name, country, DOB, etc. |
| `extract_proof_of_address(first_name, last_name)` | Python | Look up and parse utility bill / bank statement by person name -> name, address, document_type, date |
| `create_account(account_type, account_user)` | Python | Insert one row into the accounts table (called before each "account opened" message) |
| `check_pep(first_name, last_name, country, date_of_birth)` | UC function | Check person against PEP list (fuzzy match). Internal; do not show to user. |
| `check_sanctions(...)` | UC function | Check person against sanctions list (fuzzy match). Internal. |
| `check_sanctioned_country(country)` | UC function | Check passport country against sanctioned countries list (OFAC, EU, UN). Internal. |
| `check_customer_risk(sbi_code)` | UC function | Check business SBI code vs risk codes (SME path only; SBI from **DEMO_SBI_CODE**: 47.11 = low risk, 64.11 = high risk). Internal. |
| `check_fraud(first_name, last_name)` | UC function | Check person against shared fraud blacklist; returns score. Internal. |

**File naming convention:** Documents are stored in the volume with the naming convention `passport_<first_name>_<last_name>.png` and `utility_bill_<first_name>_<last_name>.png` (lowercase, spaces replaced with underscores). The tools resolve the file path automatically from the person's name.

**MLflow:** Tool calls and agent steps are logged via `mlflow.langchain.autolog()`. View traces in **Experiments -> your experiment -> Traces**.

## 1. Install dependencies

Run the cell below once; restart the kernel if needed after the first run.

In [ ]:
%pip install langchain-core langchain 'unitycatalog-ai[databricks]' 'unitycatalog-langchain[databricks]' databricks-langchain --quiet
dbutils.library.restartPython()

---

## 2. Configuration

Defaults are set in `_shared/config`. To override, change the **catalog** and **schema** widgets at the top of the notebook after running the config cell below.

**DEMO_SBI_CODE:** SBI code for the SME risk check: **47.11** = low risk (SME opened), **64.11** = high risk (SME blocked).

In [ ]:
%run ../_shared/config

In [ ]:
import os
import re
from langchain_core.tools import tool
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain.agents import create_agent

# CATALOG, SCHEMA, CATALOG_SCHEMA, VOLUME_PATH come from %run ../_shared/config

# SBI code for SME risk check — 47.11 = low risk (SME opened), 64.11 = high risk (SME blocked)
DEMO_SBI_CODE = "47.11"

# Set to False to hide tool call logs
PRINT_TOOL_MESSAGES = True

print(f"CATALOG_SCHEMA={CATALOG_SCHEMA}, VOLUME_PATH={VOLUME_PATH}, DEMO_SBI_CODE={DEMO_SBI_CODE}")

### Optional: Run setup

Run the cell below to execute **01_kyc_data_setup** in the same session. It uses **CATALOG_SCHEMA** and **VOLUME_PATH** from Configuration above. Skip if you already ran setup.

In [ ]:
%run ./01_kyc_data_setup

<cell_type>markdown</cell_type>### 3.1 Unity Catalog functions (PEP, sanctions, sanctioned countries, SBI risk, fraud)

These run as SQL in Unity Catalog; the agent calls them by name.

In [ ]:
# --- Unity Catalog functions (PEP, sanctions, sanctioned countries, SBI risk, fraud) ---
FUNCTION_NAMES = [
    f"{CATALOG_SCHEMA}.check_pep",
    f"{CATALOG_SCHEMA}.check_sanctions",
    f"{CATALOG_SCHEMA}.check_sanctioned_country",
    f"{CATALOG_SCHEMA}.check_customer_risk",
    f"{CATALOG_SCHEMA}.check_fraud",
]
uc_toolkit = UCFunctionToolkit(function_names=FUNCTION_NAMES)
uc_tools = uc_toolkit.tools

# --- Passport parsing: READ_FILES + ai_parse_document + ai_extract (path inlined; cannot be UC function) ---
def _parse_passport_sql(file_path: str) -> str:
    """Build parse SQL with path inlined (READ_FILES requires a constant)."""
    if not re.match(r"^(/Volumes/|s3://|abfss://|gs://)[\w\-\./]+$", file_path):
        return f"Error: path must start with /Volumes/, s3://, abfss://, or gs://"
    safe = file_path.replace("'", "''")
    return f"""
WITH file_data AS (
  SELECT path, content FROM READ_FILES('{safe}', format => 'binaryFile') LIMIT 1
),
parsed_docs AS (
  SELECT path, ai_parse_document(content, map('version', '2.0')) AS parsed FROM file_data
),
parsed_text AS (
  SELECT path, try_cast(parsed:error_status AS STRING) AS parse_error,
    concat_ws('\\n\\n', transform(try_cast(parsed:document:elements AS ARRAY<VARIANT>), element -> try_cast(element:content AS STRING))) AS text
  FROM parsed_docs
),
extracted AS (
  SELECT path, parse_error,
    ai_extract(text, array('first_name', 'last_name', 'country', 'date_of_birth', 'passport_issue_date', 'passport_expiry_date')) AS kyc
  FROM parsed_text
)
SELECT try_cast(kyc.first_name AS STRING) AS first_name, try_cast(kyc.last_name AS STRING) AS last_name,
  try_cast(kyc.country AS STRING) AS country, try_cast(kyc.date_of_birth AS STRING) AS date_of_birth,
  try_cast(kyc.passport_issue_date AS STRING) AS passport_issue_date, try_cast(kyc.passport_expiry_date AS STRING) AS passport_expiry_date,
  parse_error AS error FROM extracted
"""

def _normalize_name(name: str) -> str:
    """Lowercase and replace spaces with underscores for file naming."""
    return name.strip().lower().replace(" ", "_")

@tool
def extract_kyc(first_name: str, last_name: str) -> str:
    """Parse a passport document for the given person using AI. The file is looked up automatically based on the person's name using the naming convention passport_<first_name>_<last_name>.png in the volume. Returns first_name, last_name, country, date_of_birth, passport_issue_date, passport_expiry_date, and error if any."""
    file_path = f"{VOLUME_PATH}/passport_{_normalize_name(first_name)}_{_normalize_name(last_name)}.png"
    if PRINT_TOOL_MESSAGES:
        print(f"  [extract_kyc] looking up passport for {first_name} {last_name} at {file_path!r}")
    sql = _parse_passport_sql(file_path)
    if sql.startswith("Error:"):
        if PRINT_TOOL_MESSAGES:
            print(f"  [extract_kyc] validation failed: {sql}")
        return sql
    try:
        df = spark.sql(sql)
        rows = df.collect()
        if not rows:
            if PRINT_TOOL_MESSAGES:
                print("  [extract_kyc] no row returned (file not found or empty)")
            return f"No passport found for {first_name} {last_name} at {file_path}."
        row = rows[0]
        out = str(row.asDict())
        if PRINT_TOOL_MESSAGES:
            print(f"  [extract_kyc] success -> {out[:120]}{'...' if len(out) > 120 else ''}")
        return out
    except Exception as e:
        if PRINT_TOOL_MESSAGES:
            print(f"  [extract_kyc] error: {e}")
        return f"Parse error: {e}"

# --- Proof of address: same pipeline, fields name, address, document_type, document_date ---
def _parse_document_sql(file_path: str, extract_fields: list) -> str:
    """Build parse SQL for any document with READ_FILES + ai_parse_document + ai_extract (path inlined)."""
    if not re.match(r"^(/Volumes/|s3://|abfss://|gs://)[\w\-\./]+$", file_path):
        return f"Error: path must start with /Volumes/, s3://, abfss://, or gs://"
    safe = file_path.replace("'", "''")
    # Use 'doc' for the struct so it doesn't clash with CTE name 'extracted'
    fields_str = ", ".join(f"try_cast(doc.{f} AS STRING) AS {f}" for f in extract_fields)
    arr = "array(" + ", ".join(f"'{f}'" for f in extract_fields) + ")"
    return f"""
WITH file_data AS (
  SELECT path, content FROM READ_FILES('{safe}', format => 'binaryFile') LIMIT 1
),
parsed_docs AS (
  SELECT path, ai_parse_document(content, map('version', '2.0')) AS parsed FROM file_data
),
parsed_text AS (
  SELECT path, try_cast(parsed:error_status AS STRING) AS parse_error,
    concat_ws('\\n\\n', transform(try_cast(parsed:document:elements AS ARRAY<VARIANT>), element -> try_cast(element:content AS STRING))) AS text
  FROM parsed_docs
),
extracted AS (
  SELECT path, parse_error, ai_extract(text, {arr}) AS doc FROM parsed_text
)
SELECT {fields_str}, parse_error AS error FROM extracted
"""


@tool
def extract_proof_of_address(first_name: str, last_name: str) -> str:
    """Parse a Proof of Address document (utility bill or bank statement) for the given person using AI. The file is looked up automatically based on the person's name using the naming convention utility_bill_<first_name>_<last_name>.png in the volume. Returns name, address, document_type, document_date."""
    file_path = f"{VOLUME_PATH}/utility_bill_{_normalize_name(first_name)}_{_normalize_name(last_name)}.png"
    if PRINT_TOOL_MESSAGES:
        print(f"  [extract_proof_of_address] looking up utility bill for {first_name} {last_name} at {file_path!r}")
    sql = _parse_document_sql(
        file_path,
        ["name", "address", "document_type", "document_date"],
    )
    if sql.startswith("Error:"):
        if PRINT_TOOL_MESSAGES:
            print(f"  [extract_proof_of_address] validation failed: {sql}")
        return sql
    try:
        df = spark.sql(sql)
        rows = df.collect()
        if not rows:
            if PRINT_TOOL_MESSAGES:
                print("  [extract_proof_of_address] no row returned")
            return f"No proof of address found for {first_name} {last_name} at {file_path}."
        row = rows[0]
        out = str(row.asDict())
        if PRINT_TOOL_MESSAGES:
            print(f"  [extract_proof_of_address] success -> {out[:120]}{'...' if len(out) > 120 else ''}")
        return out
    except Exception as e:
        if PRINT_TOOL_MESSAGES:
            print(f"  [extract_proof_of_address] error: {e}")
        return f"Parse error: {e}"

# --- Create account: insert one row (id, account_type, account_user) before each "account opened" message ---
ACCOUNTS_TABLE = f"{CATALOG_SCHEMA}.accounts"

@tool
def create_account(account_type: str, account_user: str) -> str:
    """Create an account record (ID, account_type, account_user) in the accounts table. Call once per account before saying 'Your {account} has been opened!'. account_type e.g. 'SME Business Account', 'Investment Account', 'Personal Current Account'. account_user is the user's full name or first+last from KYC."""
    if PRINT_TOOL_MESSAGES:
        print(f"  [create_account] called: account_type={account_type!r}, account_user={account_user!r}")
    import uuid
    acc_id = str(uuid.uuid4())
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {ACCOUNTS_TABLE} (
            id STRING,
            account_type STRING,
            account_user STRING
        )
    """)
    safe_type = account_type.replace("'", "''")
    safe_user = account_user.replace("'", "''")
    spark.sql(f"INSERT INTO {ACCOUNTS_TABLE} (id, account_type, account_user) VALUES ('{acc_id}', '{safe_type}', '{safe_user}')")
    if PRINT_TOOL_MESSAGES:
        print(f"  [create_account] created id={acc_id}")
    return f"Account created: id={acc_id}, type={account_type}, user={account_user}"

# --- Combine Python tools and UC tools for the agent ---
tools = [extract_kyc, extract_proof_of_address, create_account] + uc_tools
print(f"Registered {len(tools)} tools:", [getattr(t, "name", type(t).__name__) for t in tools])

## 4. Agent (LLM + system prompt)

The system prompt defines the BankBricks conversation flow (greeting → name → account choice → identity → passport → proof of address → accounts opened → closing). The LLM and tools are wired into a LangChain agent.

In [ ]:
# MLflow: autolog + experiment for trace logging
import json
import mlflow

mlflow.langchain.autolog()
mlflow.set_tracking_uri("databricks")

_experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/kyc_agent_experiment"
_exp = mlflow.get_experiment_by_name(_experiment_name)
_experiment_id = _exp.experiment_id if _exp else mlflow.create_experiment(name=_experiment_name)
mlflow.set_experiment(experiment_id=_experiment_id)


def _screening_has_match(content):
    """Check if a UC screening function (returning TABLE) found any matches.
    Handles JSON list, CSV-in-dict wrapper, and plain CSV formats.
    Returns False for empty results (which still contain column headers)."""
    text = content.strip()
    if not text or text in ("[]", "None", "null", "{}", "''", '""'):
        return False
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return len(parsed) > 0
        if isinstance(parsed, dict):
            val = str(parsed.get("value", parsed.get("result", ""))).strip()
            if not val or val in ("[]", "None", "null"):
                return False
            try:
                inner = json.loads(val)
                if isinstance(inner, list):
                    return len(inner) > 0
            except (json.JSONDecodeError, TypeError):
                pass
            lines = [l for l in val.split("\n") if l.strip()]
            return len(lines) > 1
    except (json.JSONDecodeError, TypeError):
        pass
    lines = [l for l in text.split("\n") if l.strip()]
    return len(lines) > 1


SYSTEM_PROMPT = """You are a friendly onboarding assistant for BankBricks. Follow this conversation flow exactly. Be warm and concise.

## Conversation flow

1. **Opening (if no name yet)**  
   Say: "Hello! Welcome to BankBricks. To get started, what is your full name?"

2. **After user gives full name**  
   Extract their first name and say: "Great! Can I call you {first_name}?" (use their actual first name.)

3. **After they confirm (e.g. yes / sure)**  
   Say: "Perfect, {first_name}! What type of account would you like to open today? You can choose one or more: **Personal Current Account**, **SME Business Account**, **Investment Account**."

4. **After they pick account type(s)**  
   Say: "Excellent choice! I'll set those up for you."
   **Account-specific paths (run only for the chosen types):**
   - **SME Business Account:** Call **check_customer_risk('<DEMO_SBI_CODE>')** (assumed SBI). Do not show the raw result; keep for final outcome after Proof of Address. **If the result has risk_level 'high', do NOT create the SME Business Account later**—instead inform the user we cannot open it due to risk.
   - **Investment Account:** Walk-through path that just passes (no extra tool; proceed).
   - **Personal Current Account:** Blank pass (no check; proceed).
   Do not run check_customer_risk for Personal Current or Investment only.

5. **Identity step**  
   Say: "First, let's verify your identity. Please scan your passport using your camera. (For the demo, I'll look up your passport automatically.)"

6. **When the user confirms or says to proceed with identity verification**  
   - First reply: "Great! Verifying your document now..."
   - Then call **extract_kyc(first_name, last_name)** using the user's first name and last name. The tool will automatically find the passport file in the volume.
   - When extract_kyc returns successfully, reply: "Identity Verified! Your passport has been validated."
   - **In the same turn**, you may already ask for Proof of Address (step 7) so the user can upload while checks run. Say: "Perfect! One more step - please upload a Proof of Address document. (Recent Utility Bill or Bank Statement)"
   - **In the same turn**, call **check_pep**, **check_sanctions**, **check_sanctioned_country**, and **check_fraud** with the first_name, last_name (and country, date_of_birth for pep/sanctions; country for sanctioned-country check) from the extract_kyc result. Do not tell the user you are running these; do not show the raw results. Keep PEP/sanctions/sanctioned-country/fraud outcome for later (when you give the final "all set" or "we'll review" line after Proof of Address).

7. **Proof of Address**  
   When the user confirms or says to proceed with proof of address:
   - Call **extract_proof_of_address(first_name, last_name)** using the user's first name and last name. The tool will automatically find the utility bill file in the volume.
   - When it returns successfully, say: "Address Verified! Document accepted." Then if you have PEP/sanctions/sanctioned-country/fraud/customer-risk results: if all clear, "We've completed our checks—you're all set!" If any matches or SBI risk, "We'll need to do a quick review and will be in touch shortly." Do not mention PEP, sanctions, sanctioned country, fraud list, or SBI details.
   - Then say: "Processing your accounts.." For **each** account type the user requested: **for SME Business Account**, only if the earlier check_customer_risk result was **not** risk_level **high**—call **create_account(account_type, account_user)** and say "Your SME Business Account has been opened!". If risk_level was **high**, do NOT call create_account for SME; tell the user we cannot open the SME Business Account at this time due to our risk assessment. For **Investment Account** and **Personal Current Account**, always call **create_account(account_type, account_user)** then "Your {account} has been opened!".

8. **Closing**  
   When the user thanks you or says they're done (e.g. "That was incredibly easy. Thank you!"): reply "You're welcome, {first_name}! Can I help you with anything else today?"

## Tools (use when appropriate)

- **extract_kyc(first_name, last_name)**: Look up and parse the passport for the given person. The file is resolved automatically from the volume using the naming convention passport_<first_name>_<last_name>.png. Returns first_name, last_name, country, date_of_birth, passport_issue_date, passport_expiry_date, error. Use for identity verification.
- **extract_proof_of_address(first_name, last_name)**: Look up and parse a Proof of Address document for the given person. The file is resolved automatically from the volume using the naming convention utility_bill_<first_name>_<last_name>.png. Returns name, address, document_type, document_date, error.
- **create_account(account_type, account_user)**: Creates a row in the accounts table (id, account_type, account_user). Call once per account **before** saying "Your {account} has been opened!". Use exact account name (e.g. "SME Business Account") and user's full name.
- **check_pep(first_name, last_name, country, date_of_birth)**: Internal—use exact values from extract_kyc. Do not show this result to the user.
- **check_sanctions(first_name, last_name, country, date_of_birth)**: Internal—use exact values from extract_kyc. Do not show this result to the user.
- **check_sanctioned_country(country)**: Internal—check the passport country against the sanctioned countries list (OFAC, EU, UN). Use the country value from extract_kyc. Do not show this result to the user.
- **check_customer_risk(sbi_code)**: Internal—only use when the user is opening an **SME Business Account**. Use the assumed SBI code <DEMO_SBI_CODE> (do not ask the user for SBI). If the result has risk_level **high**, do NOT call create_account for SME Business Account; inform the user we cannot open it. Never call for Personal Current or Investment only. Do not show raw result to the user.
- **check_fraud(first_name, last_name)**: Internal—check person against the shared fraud blacklist (customers/employees; multiple financial institutions). Returns fraud_score (0–100). Use first_name, last_name from extract_kyc. Do not show raw result to the user.

Keep the tone welcoming and on-script. You can ask for Proof of Address as soon as identity is verified, while PEP/sanctions/sanctioned-country/fraud screening runs in the background."""

SYSTEM_PROMPT = SYSTEM_PROMPT.replace("<DEMO_SBI_CODE>", DEMO_SBI_CODE)

LLM_ENDPOINT = os.getenv("LLM_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1)
# LangChain 1.2+: create_agent returns a graph; invoke with {"messages": [HumanMessage(...)]}
agent = create_agent(llm, tools=tools, system_prompt=SYSTEM_PROMPT)
print(f"Agent created (LLM: {LLM_ENDPOINT}, tools: {len(tools)})")
print(f"MLflow experiment: {_experiment_name}")

## 5. Demo conversation

Multi-turn chat: the cell below runs a full onboarding flow (Hi → name → account choice → passport → proof of address → thanks). Each turn appends the user message, invokes the agent, and prints the assistant reply.

In [ ]:
import uuid
import time
from langchain_core.messages import HumanMessage

def _msg_summary(msg) -> str:
    """Short summary of a message for MLflow logging."""
    if hasattr(msg, "type"):
        content = getattr(msg, "content", "") or ""
        return f"{msg.type}: {str(content)[:200]}"
    return str(msg)[:200]

def chat(messages: list) -> tuple[list, str]:
    """Invoke agent with current message list; returns (updated messages, last assistant text)."""
    prev_len = len(messages)
    inputs = {"message_count": prev_len, "last_user": _msg_summary(messages[-1]) if messages else ""}
    with mlflow.start_span(name="kyc_agent_invoke", span_type="AGENT") as span:
        span.set_inputs(inputs)
        result = agent.invoke({"messages": messages})
        new_messages = result["messages"]
        last = new_messages[-1]
        final = (last.content if hasattr(last, "content") else str(last)) or ""
        span.set_outputs({"message_count": len(new_messages), "final_reply": final[:500]})
    # Log which tools ran this turn (so you can see check_pep / check_sanctions)
    if PRINT_TOOL_MESSAGES:
        tool_msgs = [m for m in new_messages[prev_len:] if getattr(m, "type", None) == "tool"]
        if tool_msgs:
            names = [getattr(m, "name", "?") for m in tool_msgs]
            print("  Tools executed this turn:", ", ".join(names))
            if any("check_pep" in (n or "") for n in names):
                print("  check_pep ran (PEP screening)")
            if any("check_sanctioned_country" in (n or "") for n in names):
                print("  check_sanctioned_country ran (sanctioned country screening)")
            if any(("check_sanctions" in (n or "") and "check_sanctioned_country" not in (n or "")) for n in names):
                print("  check_sanctions ran (sanctions screening)")
            if any("check_customer_risk" in (n or "") for n in names):
                print("  check_customer_risk ran (SBI risk check)")
            if any("check_fraud" in (n or "") for n in names):
                print("  check_fraud ran (fraud list score)")
    return new_messages, final

app_id = str(uuid.uuid4())
user_turns = [
    "Hi",
    "Dirk van Dijk",
    "Yes, that's fine",
    "SME Business Account and Investment Account",
    "Yes, go ahead and verify my passport",
    "Yes, please verify my proof of address",
    "That was incredibly easy. Thank you!",
]

# State tracking for screening detection
pep_flagged = sanctions_flagged = sanctioned_country_flagged = fraud_flagged = sbi_high_risk = False
screening_flagged = False
accounts_opened = []
accounts_blocked = []
requested_accounts = ["SME Business Account", "Investment Account"]
t0 = time.time()

with mlflow.start_span(name="kyc_onboarding", span_type="CHAIN") as root:
    root.set_inputs({
        "application_id": app_id,
        "applicant_name": "Dirk van Dijk",
        "requested_accounts": "SME Business Account and Investment Account",
        "persona": "demo_dirk_van_dijk",
    })

    messages = []
    for i, user_msg in enumerate(user_turns):
        turn_num = i + 1
        with mlflow.start_span(name=f"turn_{turn_num}", span_type="CHAIN") as turn:
            turn.set_inputs({"user_message": user_msg})
            messages.append(HumanMessage(content=user_msg))
            prev_len = len(messages)
            messages, reply = chat(messages)

            # Scan new messages for tool calls -> create execute_tool spans
            new_msgs = messages[prev_len:]
            call_args = {}
            for m in new_msgs:
                if getattr(m, "type", None) in ("ai", "assistant"):
                    for tc in getattr(m, "tool_calls", []) or []:
                        if tc.get("id"):
                            call_args[tc["id"]] = tc.get("args", {})
            for m in new_msgs:
                if getattr(m, "type", None) == "tool":
                    name = getattr(m, "name", "")
                    content = getattr(m, "content", "") or ""
                    if name:
                        args = call_args.get(getattr(m, "tool_call_id", ""), {})
                        with mlflow.start_span(name="execute_tool", span_type="TOOL") as ts:
                            ts.set_attribute("tool.name", name)
                            ts.set_attribute("tool.arguments", json.dumps(args, default=str))
                            ts.set_outputs({"result": content[:4000]})
                        # Screening detection
                        name_lower = name.lower()
                        if "check_pep" in name_lower:
                            if _screening_has_match(content):
                                pep_flagged = screening_flagged = True
                        elif "check_sanctioned_country" in name_lower:
                            if _screening_has_match(content):
                                sanctioned_country_flagged = screening_flagged = True
                        elif "check_sanctions" in name_lower:
                            if _screening_has_match(content):
                                sanctions_flagged = screening_flagged = True
                        elif "check_fraud" in name_lower:
                            m_fraud = re.search(r'fraud_score["\']?\s*[:=]\s*(\d+)', content)
                            if m_fraud and int(m_fraud.group(1)) > 0:
                                fraud_flagged = screening_flagged = True
                        elif "customer_risk" in name_lower:
                            if "high" in content.lower():
                                sbi_high_risk = True
                        elif "create_account" in name_lower:
                            acct_type = args.get("account_type", "")
                            if acct_type:
                                accounts_opened.append(acct_type)

            turn.set_outputs({"reply": reply[:500]})
            print(f"Assistant: {reply}\n")

    # Determine blocked accounts
    if sbi_high_risk:
        for acct in requested_accounts:
            if acct == "SME Business Account" and acct not in accounts_opened:
                accounts_blocked.append(acct)
    if screening_flagged:
        for acct in requested_accounts:
            if acct not in accounts_opened and acct not in accounts_blocked:
                accounts_blocked.append(acct)

    # Decision
    if screening_flagged:
        decision_type = "refer"
    elif accounts_blocked:
        decision_type = "conditional"
    else:
        decision_type = "approved"

    reasoning_parts = [
        "Applicant: Dirk van Dijk.",
        f"Requested accounts: {', '.join(requested_accounts)}.",
        f"PEP screening: {'MATCH' if pep_flagged else 'clear'}.",
        f"Sanctions screening: {'MATCH' if sanctions_flagged else 'clear'}.",
        f"Sanctioned country: {'MATCH' if sanctioned_country_flagged else 'clear'}.",
        f"Fraud check: {'FLAGGED' if fraud_flagged else 'clear (score 0)'}.",
        f"SBI risk: {'high' if sbi_high_risk else 'normal'}.",
    ]
    if screening_flagged:
        reasoning_parts.append("Decision: REFER for manual review.")
    elif accounts_blocked:
        reasoning_parts.append(f"Decision: CONDITIONAL. Opened: {', '.join(accounts_opened)}. Blocked: {', '.join(accounts_blocked)}.")
    else:
        reasoning_parts.append(f"Decision: APPROVED. All {len(accounts_opened)} accounts opened.")
    reasoning = " ".join(reasoning_parts)

    tags = [
        f"pep:{'match' if pep_flagged else 'clear'}",
        f"sanctions:{'match' if sanctions_flagged else 'clear'}",
        f"sanctioned_country:{'match' if sanctioned_country_flagged else 'clear'}",
        f"fraud:{'match' if fraud_flagged else 'clear'}",
        f"sbi_risk:{'high' if sbi_high_risk else 'normal'}",
        f"decision:{decision_type}",
    ]
    for acct in accounts_opened:
        tags.append(f"account_opened:{acct}")
    for acct in accounts_blocked:
        tags.append(f"account_blocked:{acct}")

    # kyc_decision span
    with mlflow.start_span(name="kyc_decision", span_type="AGENT") as dec:
        dec.set_inputs({
            "application_id": app_id,
            "applicant_name": "Dirk van Dijk",
            "screening_flagged": screening_flagged,
            "pep_match": pep_flagged,
            "sanctions_match": sanctions_flagged,
            "sanctioned_country_match": sanctioned_country_flagged,
            "fraud_match": fraud_flagged,
            "sbi_high_risk": sbi_high_risk,
            "accounts_requested": requested_accounts,
        })
        dec.set_outputs({
            "decision_type": decision_type,
            "confidence_score": 1.0,
            "reasoning": reasoning,
            "fingerprint_tags": tags,
            "accounts_opened": accounts_opened,
            "accounts_blocked": accounts_blocked,
        })

    elapsed = time.time() - t0
    root.set_outputs({
        "application_id": app_id,
        "decision_type": decision_type,
        "accounts_opened": accounts_opened,
        "accounts_blocked": accounts_blocked,
        "elapsed_s": round(elapsed, 1),
    })

print(f"\nDecision: {decision_type} | Tags: {', '.join(tags)}")
print(f"Completed in {elapsed:.1f}s")

---

## Next steps: Deploy as a Databricks App (optional)

The agent above runs inside this notebook with Spark. You can also **deploy it as a standalone Databricks App** with a chat UI and persistent conversation memory (Lakebase).

See **[03_kyc_agent_deploy]($./03_kyc_agent_deploy)** to:
1. Deploy the KYC agent to **Databricks Apps** (Responses API at `/invocations`)
2. Use it as a **subagent** from an orchestrator via `apps/kyc-onboarding-agent`
3. Enable **Lakebase-backed conversation memory** so the chat UI retains context across page reloads

The deployed app uses the same tools and system prompt — no Spark session required (uses the SQL Statement Execution API instead).

---

## Delete resources (optional)

Run the cell below to clean up all resources created by this demo. Each resource is checked before deletion — if it doesn't exist, it is skipped.

In [ ]:
# Delete resources created by this demo (tables, functions, volume, schema).
# Each resource is checked before deletion — missing resources are skipped.

_cat, _sch = CATALOG_SCHEMA.split(".", 1)

# --- Tables ---
tables = ["accounts", "pep_screening", "sanctions_screening", "sanctioned_countries", "fraud_list", "sbi_risk_codes"]
for t in tables:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG_SCHEMA}.{t}")
        print(f"  Dropped table {CATALOG_SCHEMA}.{t}")
    except Exception as e:
        print(f"  Skip table {CATALOG_SCHEMA}.{t}: {e}")

# --- UC Functions ---
functions = ["check_pep", "check_sanctions", "check_sanctioned_country", "check_customer_risk", "check_fraud"]
for f in functions:
    try:
        spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG_SCHEMA}.{f}")
        print(f"  Dropped function {CATALOG_SCHEMA}.{f}")
    except Exception as e:
        print(f"  Skip function {CATALOG_SCHEMA}.{f}: {e}")

# --- Volume ---
try:
    spark.sql(f"DROP VOLUME IF EXISTS {CATALOG_SCHEMA}.agent_volume")
    print(f"  Dropped volume {CATALOG_SCHEMA}.agent_volume")
except Exception as e:
    print(f"  Skip volume: {e}")

# --- Schema (only if empty) ---
try:
    remaining = spark.sql(f"SHOW TABLES IN {CATALOG_SCHEMA}").count()
    remaining += spark.sql(f"SHOW FUNCTIONS IN {CATALOG_SCHEMA}").filter("function NOT LIKE 'spark_catalog%'").count()
    if remaining == 0:
        spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_SCHEMA}")
        print(f"  Dropped schema {CATALOG_SCHEMA}")
    else:
        print(f"  Kept schema {CATALOG_SCHEMA} ({remaining} remaining objects)")
except Exception as e:
    print(f"  Skip schema: {e}")

print("\nCleanup complete.")